In [ ]:
! pip install langchain langchain_openai python-dotenv

In [ ]:
!pip install langchain-community pypdf

In [4]:
from dotenv import load_dotenv
load_dotenv()

True

In [16]:
from langchain_openai import OpenAIEmbeddings 
embeddings=OpenAIEmbeddings(
    model="text-embedding-3-small"
)

In [17]:
v=embeddings.embed_query('what is iti')

In [ ]:
v

In [18]:
len(v)

1536

In [19]:
import pathlib
pathes=list(pathlib.Path('./data').glob('**/*.pdf'))

In [20]:
pathes

[WindowsPath('data/monopoly.pdf'), WindowsPath('data/ticket_to_ride.pdf')]

### 2- loaders 

In [21]:
from langchain_community.document_loaders import PyPDFLoader
load_docs=[]
for path in pathes:
    loader=PyPDFLoader(path)
    load_docs.extend(loader.load())
load_docs

[Document(metadata={'producer': 'Adobe Acrobat 7.0 Paper Capture Plug-in', 'creator': 'Adobe Acrobat 7.0', 'creationdate': '2007-05-03T12:38:10-04:00', 'moddate': '2007-05-03T12:52:41-04:00', 'source': 'data\\monopoly.pdf', 'total_pages': 8, 'page': 0, 'page_label': '1'}, page_content='MONOPOLY \nProperty Trading Game from Parker Brothers" \nAGES 8+ \n2 to 8 Players \nContents: Gameboard, 3 dice, tokens, 32 houses, I2 hotels, Chance \nand Community Chest cards, Title Deed cards, play money and a Banker\'s tray. \nNow there\'s a faster way to play MONOPOLY. Choose to play by \nthe classic rules for buying, renting and selling properties or use the \nSpeed Die to get into the action faster. If you\'ve never played the classic \nMONOPOLY game, refer to the Classic Rules beginning on the next page. \nIf you already know how to play and want to use the Speed Die, just \nread the section below for the additional Speed Die rules. \nSPEED DIE RULES \nLearnins how to Play with the S~eed Die IS 

In [22]:
len(load_docs)

12

In [ ]:
from pprint import pprint
pprint(load_docs[0].page_content)

### split docs-->chunks

In [28]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


In [29]:
spiltter=RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=80,
    separators=['\n\n','\n' ,' ','']
)

In [30]:
chunks=spiltter.split_documents(load_docs)

In [32]:
len(chunks)

89

In [33]:
! pip install -qU langchain-chroma


[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [34]:
from langchain_community.vectorstores import Chroma
db=Chroma.from_documents(
   chunks,
   embeddings,
   collection_name='my_collection',
   persist_directory='./chroma_db' 
)

In [46]:
#---- search 
prompt='how to get out of jail in monopoly game ?'
chunks=db.similarity_search_with_score(prompt ,k=3)

In [45]:
chunks

[(Document(metadata={'page_label': '5', 'producer': 'Adobe Acrobat 7.0 Paper Capture Plug-in', 'page': 4, 'creator': 'Adobe Acrobat 7.0', 'moddate': '2007-05-03T12:52:41-04:00', 'source': 'data\\monopoly.pdf', 'total_pages': 8, 'creationdate': '2007-05-03T12:38:10-04:00'}, page_content='You get out of Jail by.. .(I) throwing doubles on any of your next \nthree turns; if you succeed in doing this you immediately move forward \nthe number of spaces shown by your doubles throw; even though you \nhad thrown doubles, you do not take another turn; (2) using the "Get \nOut of Jail Free" card if you have it; (3) purchasing the "Get Out of Jail'),
  0.7287832498550415),
 (Document(metadata={'moddate': '2007-05-03T12:52:41-04:00', 'creator': 'Adobe Acrobat 7.0', 'source': 'data\\monopoly.pdf', 'total_pages': 8, 'page_label': '5', 'producer': 'Adobe Acrobat 7.0 Paper Capture Plug-in', 'creationdate': '2007-05-03T12:38:10-04:00', 'page': 4}, page_content='Out of Jail Free" card if you have it; (3)

In [43]:
for chunk in chunks :
    print(chunk.page_content)
    print("_______________")

You get out of Jail by.. .(I) throwing doubles on any of your next 
three turns; if you succeed in doing this you immediately move forward 
the number of spaces shown by your doubles throw; even though you 
had thrown doubles, you do not take another turn; (2) using the "Get 
Out of Jail Free" card if you have it; (3) purchasing the "Get Out of Jail
_______________
Should you owe the Bank, instead of another player, more than you 
can pay (because of taxes or penalties) even by selling off buildings 
and mortgaging property, you must turn over all assets to the Bank. In 
this case, the Bank immediately sells by auction all property so taken, 
except buildings. A bankrupt player must immediately retire from the 
game. The last player left in the game wins.
_______________
MONOPOLY 
Property Trading Game from Parker Brothers" 
AGES 8+ 
2 to 8 Players 
Contents: Gameboard, 3 dice, tokens, 32 houses, I2 hotels, Chance 
and Community Chest cards, Title Deed cards, play money and a Banker's 

In [51]:
# ---- chatbot 
from langchain.chat_models import init_chat_model
llm=init_chat_model(
    "gpt-4.1",
    temperature=0.2
)

In [47]:
from langchain_core.prompts import PromptTemplate
rag_template=PromptTemplate(
template="""
use the context to answer the question,if the context does not contain answer 
,say i do not know.
context:{context}
question:{question}
 """,
 input_variables=['context','question']    
)

In [54]:
prompt='what is the capital of iran'
chunks=db.similarity_search(prompt ,k=3)
context="/n".join([chunk.page_content for chunk in chunks])

In [55]:
res=llm.invoke(rag_template.format(context=context,question=prompt))
print(res.content)

I do not know.


In [ ]:
#--- system prompt , tools--> [rag]

